# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/peddikotlachiranjeevi/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

**Prediction question:** identify content items whose search impressions decline in the latest 30-day period compared with the preceding 30-day period.

The starter CSV is a single export, not a historical snapshot series. Therefore this is a leakage-aware teaching feature vector, not proof of a deployable future prediction model.

Selected inputs:
- `impressions_prev_30d`: impressions in days 31?60 before export.
- `clicks_prev_30d`: clicks in days 31?60 before export.
- `sessions_prev_30d`: sessions in days 31?60 before export.
- `content_type`: content category.

The three previous-window metrics precede the latest-30-day outcome window. Numeric missing values are median-imputed, with a missingness flag retained. Categorical missing values become `unknown`, then are one-hot encoded.

`client_id` is retained separately for grouped validation and is never an input feature.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

required = [
    "content_id", "client_id", "trend_direction",
    "impressions_prev_30d", "clicks_prev_30d",
    "sessions_prev_30d", "content_type"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Required columns missing: {missing}")

# Target is a trend-derived proxy, not a verified future outcome.
y = df["trend_direction"].astype("string").str.lower().eq("down").astype("int8")

numeric_features = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
]
categorical_features = ["content_type"]

X_numeric = df[numeric_features].apply(pd.to_numeric, errors="coerce")
X_numeric = X_numeric.replace([np.inf, -np.inf], np.nan)

# Preserve missingness before median imputation.
X_missing = X_numeric.isna().astype("int8")
X_missing.columns = [f"has_missing_{c}" for c in X_missing.columns]

medians = X_numeric.median()
X_numeric = X_numeric.fillna(medians)

X_categorical = df[categorical_features].fillna("unknown").astype(str)
X_categorical = pd.get_dummies(
    X_categorical, columns=categorical_features, dtype="int8"
)

X = pd.concat([X_numeric, X_missing, X_categorical], axis=1)
groups = df["client_id"].copy()

assert len(X) == len(y) == len(groups)
assert X.notna().all().all()
assert "client_id" not in X.columns
assert "content_id" not in X.columns

print("Raw rows:", len(df))
print("Feature rows:", len(X))
print("Numeric inputs:", numeric_features)
print("Categorical inputs:", categorical_features)
print("Encoded feature columns:", X.columns.tolist())
print("Target positive rate:", round(float(y.mean()), 4))
print("Distinct client groups:", groups.nunique())
print("Missing values remaining in X:", int(X.isna().sum().sum()))
print("FEATURE VECTOR CHECK: PASS")


Raw rows: 30000
Feature rows: 30000
Numeric inputs: ['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
Categorical inputs: ['content_type']
Encoded feature columns: ['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'has_missing_impressions_prev_30d', 'has_missing_clicks_prev_30d', 'has_missing_sessions_prev_30d', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article']
Target positive rate: 0.5421
Distinct client groups: 32
Missing values remaining in X: 0
FEATURE VECTOR CHECK: PASS


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available before outcome window? |
|---|---|---|---|
| `impressions_prev_30d` | Search impressions in days 31?60 before export | Median + missing flag | Yes, according to the dictionary's window definition |
| `clicks_prev_30d` | Search clicks in days 31?60 before export | Median + missing flag | Yes, according to the dictionary's window definition |
| `sessions_prev_30d` | Sessions in days 31?60 before export | Median + missing flag | Yes, according to the dictionary's window definition |
| `content_type` | Content category | Missing becomes `unknown`; one-hot encoded | Candidate static metadata; timestamp provenance should be confirmed |
| `has_missing_*` | Indicates a missing historical numeric value | 1 means original value was missing | Derived only from selected historical inputs |

The previous-30-day window is days 31?60 back. The target compares the latest 30 days with that earlier period. The latest period is therefore not used as an input.

The CSV does not provide a row-level export timestamp or historical metadata snapshots. The availability of `content_type` at a real prediction time must be confirmed before deployment. Median values are calculated on this dataset for this teaching exercise; a production pipeline must fit imputation values on training data only and reuse them unchanged on validation/test data.


In [2]:
# Feature notes backed by the actual dataset

feature_notes = pd.DataFrame([
    {
        "feature": "impressions_prev_30d",
        "meaning": "Search impressions, days 31?60 before export",
        "missing_handling": "Median imputation + missing flag",
        "availability": "Before latest-30-day outcome window",
    },
    {
        "feature": "clicks_prev_30d",
        "meaning": "Search clicks, days 31?60 before export",
        "missing_handling": "Median imputation + missing flag",
        "availability": "Before latest-30-day outcome window",
    },
    {
        "feature": "sessions_prev_30d",
        "meaning": "Sessions, days 31?60 before export",
        "missing_handling": "Median imputation + missing flag",
        "availability": "Before latest-30-day outcome window",
    },
    {
        "feature": "content_type",
        "meaning": "Content category",
        "missing_handling": "Unknown category + one-hot encoding",
        "availability": "Static metadata candidate; timestamp confirmation needed",
    },
])

display(feature_notes)
print("\nMissing values in selected raw numeric inputs:")
print(df[numeric_features].isna().sum())
print("\nContent type distribution:")
print(df["content_type"].value_counts(dropna=False))


,feature,meaning,missing_handling,availability
0,impressions_prev_30d,"Search impressions, days 31?60 before export",Median imputation + missing flag,Before latest-30-day outcome window
1,clicks_prev_30d,"Search clicks, days 31?60 before export",Median imputation + missing flag,Before latest-30-day outcome window
2,sessions_prev_30d,"Sessions, days 31?60 before export",Median imputation + missing flag,Before latest-30-day outcome window
3,content_type,Content category,Unknown category + one-hot encoding,Static metadata candidate; timestamp confirmat...



Missing values in selected raw numeric inputs:
impressions_prev_30d    0
clicks_prev_30d         0
sessions_prev_30d       0
dtype: int64

Content type distribution:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 3. The leakage hunt

The target is `trend_direction == "down"`. The dictionary defines `trend_direction` using the latest 30-day impressions compared with the previous 30 days.

Leakage attacks:
1. Reject `trend_direction` and `trend_pct`, because they directly define or describe the target.
2. Reject latest-30-day metrics, because they are the outcome window.
3. Reject 90-day totals and rates, because their windows overlap the outcome window.
4. Reject existing-system/product flags and tiers derived from performance.
5. Keep client identifiers outside the feature matrix for grouped splitting.

The dataset has no calendar reporting date. It cannot support a genuine chronological train/test split by export date. A client-grouped split is possible, but with only 32 clients it has limitations.


In [3]:
# Explicit leakage checks

target_source = df["trend_direction"].astype("string").str.lower()
assert y.equals(target_source.eq("down").astype("int8"))

prohibited = {
    "content_id", "client_id",
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "impression_tier", "position_tier",
    "has_clicks", "has_ai_sessions", "measurable_opportunity",
    "log_impressions_90d", "log_clicks_90d",
    "log_sessions_90d", "log_ai_sessions_90d",
}

overlap = sorted(set(X.columns) & prohibited)
assert not overlap, f"LEAKAGE: prohibited columns found in X: {overlap}"

# Demonstrate the direct target leak, then keep it out of X.
leak_probe = target_source.eq("down").astype("int8")
leak_match_rate = float((leak_probe == y).mean())
assert leak_match_rate == 1.0
assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns

print("Target source: trend_direction == 'down'")
print("Deliberate target-copy leakage probe match rate:", round(leak_match_rate, 4))
print("Target-copy probe is demonstration only; it is NOT in X.")
print("Prohibited feature overlap:", overlap)
print("Latest-30-day metrics excluded: PASS")
print("Overlapping 90-day metrics excluded: PASS")
print("Label-derived columns excluded: PASS")
print("Product/performance flags excluded: PASS")
print("LEAKAGE CHECKS: PASS")


Target source: trend_direction == 'down'
Deliberate target-copy leakage probe match rate: 1.0
Target-copy probe is demonstration only; it is NOT in X.
Prohibited feature overlap: []
Latest-30-day metrics excluded: PASS
Overlapping 90-day metrics excluded: PASS
Label-derived columns excluded: PASS
Product/performance flags excluded: PASS
LEAKAGE CHECKS: PASS


## 4. What I excluded and why

| Excluded fields | Reason |
|---|---|
| `content_id`, `client_id` | Pseudonymous identifiers; grouping only, never model inputs |
| `trend_direction`, `trend_pct`, `is_declining_label` | Target source, direct label, or label-derived sibling |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | Directly overlap the latest-30-day outcome window |
| All 90-day activity totals and rates | Their trailing windows overlap the latest-30-day outcome window |
| `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | Derived from overlapping 90-day activity; `avg_position = 0` also means no data |
| `impression_tier`, `position_tier` | Performance-derived tiers; overlap outcome information |
| `has_clicks`, `has_ai_sessions`, `measurable_opportunity` | Existing preparation flags derived from 90-day performance |
| `log_*_90d` | Log transforms of overlapping 90-day activity totals |
| `provider_used`, `model_used` | Not included in the conservative vector; provenance and usefulness require separate justification |
| `competition_level`, `main_intent`, keyword fields | Excluded from this first vector; dictionary does not establish historical availability at prediction time |
| `word_count`, `char_count`, content-age and freshness tiers | Excluded until their values can be verified as of the prediction moment |
| `age_tier`, `age_tier_order`, `freshness_tier`, word/character tiers | Derived metadata; not used in the conservative vector |
| Other unused source columns | Not part of the explicitly defined feature vector |

This is intentionally conservative. Some excluded metadata may become valid features if historical snapshots prove that the values were available before the prediction window. Do not add them merely because they exist in the export.


In [4]:
# Verify the exclusion policy against the source columns

selected_source = set(numeric_features + categorical_features)
identifier_columns = {"content_id", "client_id"}
target_columns = {"trend_direction", "trend_pct", "is_declining_label"}

assert selected_source.issubset(df.columns)
assert not (selected_source & identifier_columns)
assert not (selected_source & target_columns)

excluded_source_columns = sorted(
    set(df.columns) - selected_source - identifier_columns
    - {"trend_direction"}
)

print("Selected source features:", sorted(selected_source))
print("Identifiers kept outside X:", sorted(identifier_columns))
print("Target source kept outside X: trend_direction")
print("Other excluded source columns:", len(excluded_source_columns))
for column in excluded_source_columns:
    print("-", column)

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Client groups:", groups.nunique())
print("Exclusion policy check: PASS")


Selected source features: ['clicks_prev_30d', 'content_type', 'impressions_prev_30d', 'sessions_prev_30d']
Identifiers kept outside X: ['client_id', 'content_id']
Target source kept outside X: trend_direction
Other excluded source columns: 37
- age_tier
- age_tier_order
- ai_sessions_90d
- ai_traffic_pct
- avg_position
- char_count
- char_count_tier
- clicks_90d
- clicks_last_30d
- competition
- competition_level
- content_age_days
- cpc
- ctr
- days_since_last_update
- days_with_impressions
- days_with_sessions
- engaged_sessions_90d
- engagement_rate
- freshness_tier
- impression_tier
- impressions_90d
- impressions_last_30d
- main_intent
- model_used
- pageviews_90d
- position_tier
- provider_used
- scroll_events_90d
- scroll_rate
- search_volume
- sessions_90d
- sessions_last_30d
- trend_pct
- users_90d
- word_count
- word_count_tier

Feature matrix shape: (30000, 9)
Target shape: (30000,)
Client groups: 32
Exclusion policy check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.